In [28]:
import duckdb

con = duckdb.connect()
con.execute("""
    CREATE VIEW financials AS
    SELECT * FROM read_csv_auto('/Users/shivangi/Desktop/Sportswear Financial Performance Analysis_adida_vs _Puma/data/financials_clean.csv')
""")

In [29]:
con.sql("""
    SELECT company, year, value AS revenue
    FROM financials
    WHERE metric = 'revenue'
    ORDER BY company, year
""").df()

,company,year,revenue
0,adidas,2022,2.251100e+10
1,adidas,2023,2.142700e+10
2,adidas,2024,2.368300e+10
3,adidas,2025,2.481100e+10
4,puma,2022,8.465100e+09
5,puma,2023,8.601700e+09
6,puma,2024,8.817200e+09
7,puma,2025,7.296200e+09


In [30]:
con.sql("""
WITH revenue AS (
    SELECT company, year, value AS revenue
    FROM financials
    WHERE metric = 'revenue'
)
SELECT
    company,
    year,
    revenue,
    LAG(revenue) OVER (PARTITION BY company ORDER BY year) AS prior_year_revenue,
    ROUND(
        (revenue - LAG(revenue) OVER (PARTITION BY company ORDER BY year))
        / LAG(revenue) OVER (PARTITION BY company ORDER BY year) * 100
    , 1) AS revenue_growth_pct
FROM revenue
ORDER BY company, year
""").df()

,company,year,revenue,prior_year_revenue,revenue_growth_pct
0,adidas,2022,2.251100e+10,NaN,NaN
1,adidas,2023,2.142700e+10,2.251100e+10,-4.8
2,adidas,2024,2.368300e+10,2.142700e+10,10.5
3,adidas,2025,2.481100e+10,2.368300e+10,4.8
4,puma,2022,8.465100e+09,NaN,NaN
5,puma,2023,8.601700e+09,8.465100e+09,1.6
6,puma,2024,8.817200e+09,8.601700e+09,2.5
7,puma,2025,7.296200e+09,8.817200e+09,-17.3


In [31]:
con.sql("""
WITH pivoted AS (
    SELECT
        company,
        year,
        MAX(CASE WHEN metric = 'revenue' THEN value END) AS revenue,
        MAX(CASE WHEN metric = 'gross_profit' THEN value END) AS gross_profit
    FROM financials
    GROUP BY company, year
)
SELECT
    company,
    year,
    revenue,
    gross_profit,
    ROUND(gross_profit / revenue * 100, 1) AS gross_margin_pct
FROM pivoted
ORDER BY company, year
""").df()

,company,year,revenue,gross_profit,gross_margin_pct
0,adidas,2022,2.251100e+10,1.064400e+10,47.3
1,adidas,2023,2.142700e+10,1.018300e+10,47.5
2,adidas,2024,2.368300e+10,1.202500e+10,50.8
3,adidas,2025,2.481100e+10,1.280500e+10,51.6
4,puma,2022,8.465100e+09,3.902700e+09,46.1
5,puma,2023,8.601700e+09,3.986600e+09,46.3
6,puma,2024,8.817200e+09,3.997800e+09,45.3
7,puma,2025,7.296200e+09,3.279600e+09,44.9


In [32]:
con.sql("""
WITH pivoted AS (
    SELECT
        company,
        year,
        MAX(CASE WHEN metric = 'revenue' THEN value END)          AS revenue,
        MAX(CASE WHEN metric = 'operating_income' THEN value END) AS operating_income,
        MAX(CASE WHEN metric = 'net_income' THEN value END)       AS net_income
    FROM financials
    GROUP BY company, year
)
SELECT
    company,
    year,
    revenue,
    operating_income,
    net_income,
    ROUND(operating_income / revenue * 100, 1) AS ebit_margin_pct,
    ROUND(net_income / revenue * 100, 1)       AS net_margin_pct
FROM pivoted
ORDER BY company, year
""").df()

,company,year,revenue,operating_income,net_income,ebit_margin_pct,net_margin_pct
0,adidas,2022,2.251100e+10,7.290000e+08,6.120000e+08,3.2,2.7
1,adidas,2023,2.142700e+10,2.790000e+08,-7.500000e+07,1.3,-0.4
2,adidas,2024,2.368300e+10,1.291000e+09,7.640000e+08,5.5,3.2
3,adidas,2025,2.481100e+10,2.061000e+09,1.340000e+09,8.3,5.4
4,puma,2022,8.465100e+09,6.405000e+08,3.535000e+08,7.6,4.2
5,puma,2023,8.601700e+09,6.215000e+08,3.049000e+08,7.2,3.5
6,puma,2024,8.817200e+09,5.486000e+08,2.816000e+08,6.2,3.2
7,puma,2025,7.296200e+09,-3.574000e+08,-6.455000e+08,-4.9,-8.8


In [33]:
con.sql("""
WITH pivoted AS (
    SELECT
        company,
        year,
        MAX(CASE WHEN metric = 'operating_income'     THEN value END) AS operating_income,
        MAX(CASE WHEN metric = 'total_assets'         THEN value END) AS total_assets,
        MAX(CASE WHEN metric = 'current_liabilities'  THEN value END) AS current_liabilities,
        MAX(CASE WHEN metric = 'total_debt'           THEN value END) AS total_debt,
        MAX(CASE WHEN metric = 'equity'               THEN value END) AS equity,
        MAX(CASE WHEN metric = 'interest_expense'     THEN value END) AS interest_expense
    FROM financials
    GROUP BY company, year
)
SELECT
    company,
    year,
    ROUND(operating_income / (total_assets - current_liabilities) * 100, 1) AS roce_pct,
    ROUND(total_debt / equity, 2)                                           AS debt_to_equity,
    ROUND(operating_income / ABS(interest_expense), 1)                      AS interest_coverage_x
FROM pivoted
ORDER BY company, year
""").df()

,company,year,roce_pct,debt_to_equity,interest_coverage_x
0,adidas,2022,6.6,1.29,5.3
1,adidas,2023,2.8,1.21,1.7
2,adidas,2024,11.7,1.02,5.8
3,adidas,2025,18.5,0.96,9.0
4,puma,2022,16.3,0.63,5.2
5,puma,2023,15.1,0.71,6.2
6,puma,2024,12.8,0.61,4.7
7,puma,2025,-11.0,1.47,-3.2


In [39]:
con.sql("""
WITH pivoted AS (
    SELECT company, year,
        MAX(CASE WHEN metric = 'total_debt' THEN value END) AS total_debt,
        MAX(CASE WHEN metric = 'equity' THEN value END) AS equity,
        MAX(CASE WHEN metric = 'interest_expense' THEN value END) AS interest_expense,
        MAX(CASE WHEN metric = 'operating_income' THEN value END) AS operating_income
    FROM financials GROUP BY company, year
)
SELECT company, year,
    ROUND(total_debt/1e6,0) AS debt_m,
    ROUND(equity/1e6,0) AS equity_m,
    ROUND(total_debt/equity,2) AS debt_to_equity,
    ROUND(operating_income/ABS(interest_expense),1) AS interest_coverage_x,
    ROUND((equity-LAG(equity) OVER(PARTITION BY company ORDER BY year))/1e6,0) AS chg_equity_m,
    ROUND((total_debt-LAG(total_debt) OVER(PARTITION BY company ORDER BY year))/1e6,0) AS chg_debt_m
FROM pivoted ORDER BY company, year
""").df()

,company,year,debt_m,equity_m,debt_to_equity,interest_coverage_x,chg_equity_m,chg_debt_m
0,adidas,2022,6459.0,4991.0,1.29,5.3,NaN,NaN
1,adidas,2023,5563.0,4580.0,1.21,1.7,-411.0,-896.0
2,adidas,2024,5587.0,5476.0,1.02,5.8,896.0,24.0
3,adidas,2025,5556.0,5776.0,0.96,9.0,300.0,-31.0
4,puma,2022,1558.0,2472.0,0.63,5.2,NaN,NaN
5,puma,2023,1804.0,2553.0,0.71,6.2,82.0,247.0
6,puma,2024,1719.0,2828.0,0.61,4.7,274.0,-86.0
7,puma,2025,2582.0,1761.0,1.47,-3.2,-1067.0,864.0


In [34]:
con.sql("""
WITH pivoted AS (
    SELECT company, year,
        MAX(CASE WHEN metric = 'net_income' THEN value END) AS net_income,
        MAX(CASE WHEN metric = 'operating_cash_flow' THEN value END) AS ocf,
        MAX(CASE WHEN metric = 'receivables' THEN value END) AS receivables,
        MAX(CASE WHEN metric = 'inventory' THEN value END) AS inventory,
        MAX(CASE WHEN metric = 'payables' THEN value END) AS payables
    FROM financials GROUP BY company, year
)
SELECT company, year,
    ROUND(net_income/1e6,0) AS net_income_m,
    ROUND(ocf/1e6,0) AS ocf_m,
    ROUND((ocf-net_income)/1e6,0) AS cash_minus_profit_m,
    ROUND((receivables - LAG(receivables) OVER w)/1e6,0) AS chg_receivables_m,
    ROUND((inventory - LAG(inventory) OVER w)/1e6,0) AS chg_inventory_m,
    ROUND((payables - LAG(payables) OVER w)/1e6,0) AS chg_payables_m
FROM pivoted
WINDOW w AS (PARTITION BY company ORDER BY year)
ORDER BY company, year
""").df()

,company,year,net_income_m,ocf_m,cash_minus_profit_m,chg_receivables_m,chg_inventory_m,chg_payables_m
0,adidas,2022,612.0,-479.0,-1091.0,NaN,NaN,NaN
1,adidas,2023,-75.0,2550.0,2625.0,-623.0,-1448.0,-632.0
2,adidas,2024,764.0,2910.0,2146.0,507.0,464.0,820.0
3,adidas,2025,1340.0,751.0,-589.0,222.0,843.0,-186.0
4,puma,2022,354.0,418.0,65.0,NaN,NaN,NaN
5,puma,2023,305.0,654.0,349.0,54.0,-440.0,-235.0
6,puma,2024,282.0,695.0,413.0,128.0,212.0,394.0
7,puma,2025,-646.0,-319.0,326.0,-333.0,45.0,-622.0


In [35]:
con.sql("""
SELECT metric, value
FROM financials
WHERE company = 'adidas' AND year = 2025
  AND metric IN ('operating_income','total_assets','current_liabilities','total_debt','equity')
""").df()

,metric,value
0,current_liabilities,9.094000e+09
1,equity,5.776000e+09
2,operating_income,2.061000e+09
3,total_assets,2.026200e+10
4,total_debt,5.556000e+09


In [36]:
con.sql("""
WITH pivoted AS
(
SELECT 
company,year,
 MAX(CASE WHEN metric = 'revenue'   THEN value END) AS revenue,
        MAX(CASE WHEN metric = 'cost_of_revenue' THEN value END) AS cost_of_revenue,
        MAX(CASE WHEN metric = 'receivables'     THEN value END) AS receivables,
        MAX(CASE WHEN metric = 'inventory'       THEN value END) AS inventory,
        MAX(CASE WHEN metric = 'payables'        THEN value END) AS payables
    FROM financials
    GROUP BY company, year
),
days AS (
    SELECT
        company,
        year,
        receivables / revenue * 365         AS dso,
        inventory / cost_of_revenue * 365   AS dio,
        payables / cost_of_revenue * 365    AS dpo
    FROM pivoted
)
SELECT
    company,
    year,
    ROUND(dso, 1)             AS dso_days,
    ROUND(dio, 1)             AS dio_days,
    ROUND(dpo, 1)             AS dpo_days,
    ROUND(dso + dio - dpo, 1) AS cash_conversion_cycle_days
FROM days
ORDER BY company, year
""").df()

,company,year,dso_days,dio_days,dpo_days,cash_conversion_cycle_days
0,adidas,2022,41.0,183.7,89.4,135.3
1,adidas,2023,32.5,146.9,73.9,105.5
2,adidas,2024,37.2,156.2,96.9,96.5
3,adidas,2025,38.8,177.3,88.5,127.6
4,puma,2022,45.9,179.4,138.8,86.5
5,puma,2023,47.5,142.5,118.6,71.3
6,puma,2024,51.6,167.0,157.1,61.6
7,puma,2025,45.7,187.1,115.5,117.2


In [37]:
con.sql("""
WITH pivoted AS (
    SELECT
        company,
        year,
        MAX(CASE WHEN metric = 'total_assets'      THEN value END) AS total_assets,
        MAX(CASE WHEN metric = 'total_liabilities' THEN value END) AS total_liabilities,
        MAX(CASE WHEN metric = 'total_equity' THEN value END) AS total_equity
    FROM financials
    GROUP BY company, year
)
SELECT
    company,
    year,
    ROUND(total_assets / 1e6, 0)                                AS assets_m,
    ROUND((total_liabilities + total_equity) / 1e6, 0)                AS liabilities_plus_equity_m,
    ROUND((total_assets - total_liabilities - total_equity) / 1e6, 0) AS difference_m
FROM pivoted
ORDER BY company, year
""").df()

,company,year,assets_m,liabilities_plus_equity_m,difference_m
0,adidas,2022,20296.0,20296.0,0.0
1,adidas,2023,18020.0,18020.0,0.0
2,adidas,2024,20655.0,20655.0,0.0
3,adidas,2025,20262.0,20262.0,0.0
4,puma,2022,6773.0,6773.0,0.0
5,puma,2023,6640.0,6640.0,0.0
6,puma,2024,7141.0,7141.0,0.0
7,puma,2025,6455.0,6455.0,0.0


In [40]:
con.sql("""
WITH revenue AS (
    SELECT company, year,
        MAX(CASE WHEN metric = 'revenue' THEN value END) AS revenue
    FROM financials GROUP BY company, year
),
growth AS (
    SELECT company, year, revenue,
        (revenue - LAG(revenue) OVER (PARTITION BY company ORDER BY year))
            / LAG(revenue) OVER (PARTITION BY company ORDER BY year) AS growth_rate
    FROM revenue
),
avg_growth AS (
    -- naive assumption: average growth rate from all prior years (2022-2024) per company
    SELECT company, AVG(growth_rate) AS avg_hist_growth
    FROM growth
    WHERE year < 2025
    GROUP BY company
)
SELECT
    g.company,
    r24.revenue AS revenue_2024_actual,
    a.avg_hist_growth,
    ROUND(r24.revenue * (1 + a.avg_hist_growth) / 1e6, 0) AS naive_budget_2025_m,
    ROUND(g.revenue / 1e6, 0) AS actual_2025_m,
    ROUND((g.revenue - r24.revenue * (1 + a.avg_hist_growth)) / 1e6, 0) AS variance_m,
    ROUND((g.revenue - r24.revenue * (1 + a.avg_hist_growth)) / (r24.revenue * (1 + a.avg_hist_growth)) * 100, 1) AS variance_pct
FROM growth g
JOIN avg_growth a ON g.company = a.company
JOIN revenue r24 ON r24.company = g.company AND r24.year = 2024
WHERE g.year = 2025
""").df()

,company,revenue_2024_actual,avg_hist_growth,naive_budget_2025_m,actual_2025_m,variance_m,variance_pct
0,puma,8.817200e+09,0.020595,8999.0,7296.0,-1703.0,-18.9
1,adidas,2.368300e+10,0.028567,24360.0,24811.0,451.0,1.9
